# STEER on MultiBench

One pretrained model per dataset, probed at 15 preferences
`λ = (λ_R, λ_U₁, λ_U₂)`. λ\* is selected on validation across seeds using consesus, then every seed is
re-probed at that single λ\* and the score reported. MOSEI is scored on the non-neutral
test set; UR-FUNNY and AV-MNIST on top-1.



## Setup

In [ ]:
%pip install -q torch scikit-learn numpy scipy einops matplotlib tqdm

In [1]:
import os, sys, json, glob, statistics as st
sys.path.insert(0, os.getcwd())
import numpy as np, torch

import pareto_ssl.multibench.probe as P
from pareto_ssl.multibench.consensus_pick import consensus

P._READOUT[0] = "dim"                 # read-out used throughout the paper
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = list(range(42, 47))
ARM  = {"mosei": "lc025_m5", "humor": "lc025_m5", "avmnist": "lc025_m5"}
PFIT = {"mosei": "valid", "humor": "valid", "avmnist": "valid"}
CACHE = "runs/probe_cache"   
os.makedirs(CACHE, exist_ok=True)
CKPT = ("pareto_ssl/multibench/results/Output/multibench/{ds}/approach5/"
        "simclr_single_per_batch_filmsimplex_enc_decomp_R_loralinear_{arm}_base_seed{s}")

def ckpt(ds, seed):
    return CKPT.format(ds=ds, arm=ARM[ds], s=seed)
print(f"torch {torch.__version__} | {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'cpu'}")

torch 2.6.0+cu124 | NVIDIA H100 NVL


In [2]:
def sweep(ds, seed, fixed=None):
    """Probe all 15 preferences; fixed=λ evaluates at that λ instead of the argmax."""
    tag = f"{CACHE}/{ds}_{'fixed' if fixed else 'sweep'}_seed{seed}.json"
    if os.path.exists(tag):
        return json.load(open(tag))
    P.reset_readout_state()
    P._PROBE_FIT[0] = PFIT[ds]
    P._FIXED_LAM[0] = tuple(fixed) if fixed else None
    out = P.run_validation(enc_dir=ckpt(ds, seed), dataset=ds, approach=5,
                           enc_dim=40, proj_dim=64, batch_size=128, device=DEVICE,
                           enc_ckpt_v=None, enc_ckpt_t=None,
                           film_mode="simplex_enc_decomp_R", modality="both")
    P._FIXED_LAM[0] = None
    P._PROBE_FIT[0] = "valid"
    json.dump(out, open(tag, "w"))
    return out


def score(ds, out):
    """MOSEI is reported on the non-neutral test set; the others on top-1."""
    if ds == "mosei":
        return (out.get("acc2") or {})["non_neutral"]
    return out["test_top1_acc"]


def run(ds):
    curves = {}
    for s in SEEDS:
        o = sweep(ds, s)
        curves[s] = dict(zip([tuple(l) for l in o["lambdas"]], o["val_acc_pct"]))
    lam = consensus(curves, list(curves[SEEDS[0]]), SEEDS)[0]
    return lam, [score(ds, sweep(ds, s, fixed=lam)) for s in SEEDS]

## CMU-MOSEI

In [3]:
# how the checkpoints were produced (not run here)
# python pareto_ssl/multibench/benchmark_multibench.py \\
#     --approach 5 --method simclr_single_per_batch --dataset mosei \\
#     --film_mode simplex_enc_decomp_R --arch factorcl_based \\
#     --enc_dim 40 --proj_dim 64 --epochs 200 --lr 3e-4 --batch_size 128 \\
#     --lora_branches 2 --lora_rank 4 --lora_layers 3 --lora_lr_mult 10 \\
#     --preference_schedule annealed --num_preferences 5 --prefs_per_batch 5 \\
#     --lam_club 0.25 --seed 42 --device cuda
for split in ("train", "valid", "test"):
    print(f"{split:<6} n = {len(P._build_probe_loader('mosei', split, 128).dataset):>6}")
print(f"checkpoints: {len(SEEDS)} seeds, 5 preferences per minibatch")

train  n =  16265
valid  n =   1869
test   n =   4643
checkpoints: 5 seeds, 5 preferences per minibatch


In [4]:
lam_mosei, test_mosei = run("mosei")
print(f"λ* = {tuple(round(x, 2) for x in lam_mosei)}")
print(f"test = {st.mean(test_mosei):.2f} ± {st.stdev(test_mosei):.2f}")

λ* = (0.0, 0.25, 0.75)
test = 73.11 ± 0.95


## UR-FUNNY

In [5]:
# how the checkpoints were produced (not run here)
# python pareto_ssl/multibench/benchmark_multibench.py \\
#     --approach 5 --method simclr_single_per_batch --dataset humor \\
#     --film_mode simplex_enc_decomp_R --arch factorcl_based \\
#     --enc_dim 40 --proj_dim 64 --epochs 200 --lr 3e-4 --batch_size 128 \\
#     --lora_branches 2 --lora_rank 4 --lora_layers 3 --lora_lr_mult 10 \\
#     --preference_schedule annealed --num_preferences 5 --prefs_per_batch 5 \\
#     --lam_club 0.25 --seed 42 --device cuda
for split in ("train", "valid", "test"):
    print(f"{split:<6} n = {len(P._build_probe_loader('humor', split, 128).dataset):>6}")
print(f"checkpoints: {len(SEEDS)} seeds, 5 preferences per minibatch")

train  n =   8074
valid  n =   1034
test   n =   1058
checkpoints: 5 seeds, 5 preferences per minibatch


In [6]:
lam_humor, test_humor = run("humor")
print(f"λ* = {tuple(round(x, 2) for x in lam_humor)}")
print(f"test = {st.mean(test_humor):.2f} ± {st.stdev(test_humor):.2f}")

λ* = (0.0, 0.5, 0.5)
test = 58.07 ± 1.08


## AV-MNIST

In [7]:
# how the checkpoints were produced (not run here)
# python pareto_ssl/multibench/benchmark_multibench.py \\
#     --approach 5 --method simclr_single_per_batch --dataset avmnist \\
#     --film_mode simplex_enc_decomp_R --arch factorcl_based \\
#     --enc_dim 40 --proj_dim 64 --epochs 200 --lr 3e-4 --batch_size 128 \\
#     --lora_branches 2 --lora_rank 4 --lora_layers 3 --lora_lr_mult 10 \\
#     --preference_schedule annealed --num_preferences 5 --prefs_per_batch 5 \\
#     --lam_club 0.25 --seed 42 --device cuda
for split in ("train", "valid", "test"):
    print(f"{split:<6} n = {len(P._build_probe_loader('avmnist', split, 128).dataset):>6}")
print(f"checkpoints: {len(SEEDS)} seeds, 5 preferences per minibatch")

train  n =  55000
valid  n =   5000
test   n =  10000
checkpoints: 5 seeds, 5 preferences per minibatch


In [8]:
lam_avmnist, test_avmnist = run("avmnist")
print(f"λ* = {tuple(round(x, 2) for x in lam_avmnist)}")
print(f"test = {st.mean(test_avmnist):.2f} ± {st.stdev(test_avmnist):.2f}")

λ* = (1.0, 0.0, 0.0)
test = 69.72 ± 0.19


## Summary

In [9]:
print(f"{'dataset':<12}{'λ*':>22}{'score':>16}")
print("-" * 50)
for name, lam, t in (("CMU-MOSEI", lam_mosei, test_mosei),
                     ("UR-FUNNY", lam_humor, test_humor),
                     ("AV-MNIST", lam_avmnist, test_avmnist)):
    print(f"{name:<12}{str(tuple(round(x, 2) for x in lam)):>22}"
          f"{st.mean(t):>11.2f} ± {st.stdev(t):.2f}")
print("\nMOSEI: accuracy on the non-neutral test set. "
      "UR-FUNNY: accuracy. AV-MNIST: top-1.")

dataset                         λ*           score
--------------------------------------------------
CMU-MOSEI        (0.0, 0.25, 0.75)      73.11 ± 0.95
UR-FUNNY           (0.0, 0.5, 0.5)      58.07 ± 1.08
AV-MNIST           (1.0, 0.0, 0.0)      69.72 ± 0.19

MOSEI: accuracy on the non-neutral test set. UR-FUNNY: accuracy. AV-MNIST: top-1.
